In [ ]:
# ===== nb2 mu-LADDER (BIG, chunked) — CONFIG (edit ONLY this cell) ========

REPO_URL = "https://github.com/Avi161/ACSolverX.git"
BRANCH   = "research/w5/stable-ac-escape"   # must match the actual git branch
REPO_DIR = "ACSolverX"
CLONE       = True
UPDATE_REPO = True           # git reset --hard so a RESTART pulls latest push

MOUNT_DRIVE = True           # mirror results/stable_ac/mu_scan/*.jsonl to Drive
                             # every few minutes + at the end, and seed them BACK
                             # on a fresh VM so resume continues where the last
                             # session stopped (writes always land locally first)
DRIVE_DIR   = "/content/drive/MyDrive/acsolverx_results/stable_ac_mu_scan"

# --- ladder knobs ----------------------------------------------------------
# Budgets are DETERMINISTIC (rungs + max_orbits — the ladder's nodes_explored):
# a class stopped at its budget is the same result on any CPU. The time knob
# is only a runaway backstop; a row it catches is flagged timed_out and is the
# only machine-dependent kind. Canonicalisation runs on the numba twin
# (autcanon_fast, ~15x measured end-to-end, rows identical — test-pinned);
# the verifier re-checks every row with the slow pure-Python aut_canon.
DATA    = "data/ms_unsolved_reps/aca_124.csv"
RUNGS   = 256         # rung ceiling (deterministic budget #1)
BEAM    = 64          # K lowest-mu new orbits kept per rung (uphill allowed)
CAP     = 24          # forwarded default_cap; the admissibility ceiling is
                      # cov.REJECT_LEN = 239 (structural, never a length prior)
STOP_MU = 12          # 12, NOT 13 — mu=13 starters (aca_115) must climb, not be
                      # skipped at rung 0; a mu<=12 hit is a LEAD (MU_CRITERION.md
                      # verification bar applies; on aca_115 / the AK(3) orbit it
                      # is a PRESUMED BUG until independently reproduced)
TIME_PER_CLASS_S = 14400   # 4h RUNAWAY BACKSTOP only, not the budget
MAX_ORBITS       = 150_000 # deterministic budget #2 (also the RAM brake)

HIGH_SPEEDUP = True   # numba canonicalizer (autcanon_fast, ~15x, rows
                      # identical — test-pinned). False = force the pure-
                      # Python aut_canon path; result-neutral either way, so
                      # files resume across the two modes.

CHUNKS      = 5       # chunk processes; match to the session's CPU count
CHUNK_INDEX = None    # None = all chunks as parallel processes here; i = only
                      # chunk i (one per parallel Colab session); a LIST like
                      # [1, 6, 11, ...] = those chunks as parallel processes here
                      # (5 sessions x 8 cores: CHUNKS=40, session s runs
                      # [s, s+5, s+10, ..., s+35])
NAMES       = None    # None = all 124 classes; a list restricts the run
                      # (smoke test: ["aca_0", "aca_1", "aca_115"])


In [ ]:
# ==================== SETUP (clone / pull / install) ======================
# Zero search nodes — but canonicalisation runs on the numba twin
# (autcanon_fast), so numba/numpy are needed; the pure-Python path is only
# the automatic fallback (same rows, ~15x slower).
import os, sys, subprocess

def sh(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-2000:])
    if p.returncode != 0 and p.stderr: print("STDERR:", p.stderr[-2000:])

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Colab:", IN_COLAB)

if IN_COLAB:
    BASE = "/content"
    os.chdir(BASE)                       # anchor so re-runs never nest the clone
    if not os.path.isdir(REPO_DIR):
        if CLONE:
            sh(f"git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}")
    elif UPDATE_REPO:
        sh(f"cd {REPO_DIR} && git fetch --depth 1 origin {BRANCH} && git reset --hard FETCH_HEAD")
    sh(f"cd {REPO_DIR} && git log -1 --oneline")
    sh("pip -q install numba numpy")
    REPO_ROOT = os.path.join(BASE, REPO_DIR)
else:
    # local: walk up from cwd to the repo root (dir holding experiments/ + data/)
    REPO_ROOT = os.getcwd()
    while REPO_ROOT != "/" and not (
        os.path.isdir(os.path.join(REPO_ROOT, "experiments"))
        and os.path.isdir(os.path.join(REPO_ROOT, "data"))
    ):
        REPO_ROOT = os.path.dirname(REPO_ROOT)

os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("repo root:", REPO_ROOT)

# a `git reset --hard` rewrites .py files but sys.modules keeps the OLD module
# objects -- drop them so RUN imports what SETUP just fetched (pull != reload)
import importlib
for _m in [m for m in sys.modules if m == "experiments" or m.startswith("experiments.")]:
    del sys.modules[_m]
importlib.invalidate_caches()


# --- Drive: mount + seed-back (fresh VM -> local resume state) ------------
import glob, shutil
LOCAL_OUT = os.path.join(REPO_ROOT, "results", "stable_ac", "mu_scan")
os.makedirs(LOCAL_OUT, exist_ok=True)
if IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    for src in glob.glob(os.path.join(DRIVE_DIR, "*.jsonl")):
        dst = os.path.join(LOCAL_OUT, os.path.basename(src))
        # the bigger file wins: local mid-run state beats a stale mirror,
        # and on a fresh VM the mirror beats the (absent/empty) local file
        if not os.path.exists(dst) or os.path.getsize(dst) < os.path.getsize(src):
            shutil.copyfile(src, dst)
            print("seeded from Drive:", os.path.basename(src))


In [ ]:
# ==================== RUN =================================================
from experiments.stable_ac.cov.mu_ladder_big import run

# mirror local jsonls -> Drive every 3 min (and once at the end). Whole-file
# copies of append-only jsonls: a torn tail line is repaired on resume. The
# thread never prints (background threads must not).
import threading, time as _time
def _sync_to_drive():
    if not (IN_COLAB and MOUNT_DRIVE):
        return
    for src in glob.glob(os.path.join(LOCAL_OUT, "*.jsonl")):
        dst = os.path.join(DRIVE_DIR, os.path.basename(src))
        # size-monotonic: an append-only jsonl only ever grows, so never
        # overwrite a bigger Drive copy with a smaller local one (a session
        # holding a seeded STALE copy of another session's chunk must not
        # clobber the owner's fresh mirror)
        if not os.path.exists(dst) or os.path.getsize(dst) < os.path.getsize(src):
            tmp = dst + ".tmp"
            shutil.copyfile(src, tmp)
            os.replace(tmp, dst)
def _mirror_loop():
    while not _mirror_stop.wait(180):
        try: _sync_to_drive()
        except Exception: pass                 # transient Drive hiccup: next tick
_mirror_stop = threading.Event()
threading.Thread(target=_mirror_loop, daemon=True).start()

run(data=DATA, rungs=RUNGS, beam=BEAM, cap=CAP, stop_mu=STOP_MU,
    time_per_class_s=TIME_PER_CLASS_S, max_orbits=MAX_ORBITS,
    chunks=CHUNKS, chunk_index=CHUNK_INDEX, names=NAMES,
    high_speedup=HIGH_SPEEDUP)

_mirror_stop.set()
_sync_to_drive()                               # final sync, incl. the last rows
if IN_COLAB and MOUNT_DRIVE:
    print("mirrored to", DRIVE_DIR)


In [ ]:
# ==================== MERGE (after ALL chunks finish) =====================
# Concatenates the completed _c{i}of{N} chunk jsonls (summary + orbits) into
# the canonical unchunked pair. It REFUSES to run while any chunk is missing
# classes (row sets are re-derived from the CSV stride, never trusted from
# the files) or when the target already exists, so running it early or twice
# is safe. Then verify EVERY recorded state independently:
#   .venv/bin/python3 -m experiments.stable_ac.cov.verify_mu_ladder \
#       results/stable_ac/mu_scan/<merged>.jsonl --level full
# (--level replay skips the aut_canon re-derivation and is much faster)
if CHUNKS > 1:
    from experiments.stable_ac.cov.mu_ladder_big import merge_chunks
    merge_chunks(data=DATA, rungs=RUNGS, beam=BEAM, cap=CAP, stop_mu=STOP_MU,
                 time_per_class_s=TIME_PER_CLASS_S, max_orbits=MAX_ORBITS,
                 chunks=CHUNKS, names=NAMES)
    _sync_to_drive()                           # mirror the merged files too
